In [6]:
import os
import feedparser
import pytz

from datetime import datetime, timedelta
from dataclasses import dataclass
from time import mktime

tz = pytz.timezone(os.getenv('TIMEZONE', 'UTC'))
now = datetime.now(tz)
# TODO: handle first day of month
# last_day = now.replace(day=now.day - 1)

banned_tags = set([
    'government & policy',
    'smartglasses',
    'north korea',
    'space',
    'lawsuit',
    'geopolitics',
    'space & astrophysics'
])

@dataclass
class Feed:
    name: str
    url: str
    priority: int

@dataclass
class Article:
    feed: Feed
    title: str
    description: str
    summary: str
    link: str
    published: datetime
    authors: list
    tags: list
    priority: int

feeds = [
    Feed(name="Email", url="feed://feedyour.email/feeds/34e875tdgt84hw6pl3fj8ywp.atom", priority=1) # works with crawl4ai
]

articles = []

for feed in feeds:
    news = feedparser.parse(feed.url)

    for entry in news.entries:
        published = entry.get("published_parsed", None)

        if published:
            published = tz.localize(datetime.fromtimestamp(mktime(published)))
            if now - published > timedelta(hours=24):
                continue
        else:
            published = now

        tags = set([tag.get("term", "").lower() for tag in entry.get("tags", [])])
        if banned_tags.intersection(tags):
            continue

        title = entry.get("title", "")
        full_text = entry.get("content", [{}])[0].get("value", entry.get("summary", ""))
        description = entry.get("summary", "")
        link = entry.get("link", "")
        authors = [author.get("name", "") for author in entry.get("authors", [])]

        article = Article(
            feed=feed,
            title=title,
            description=description,
            summary=None,
            link=link,
            published=published,
            authors=authors,
            tags=list(tags),
            priority=feed.priority
        )

        articles.append(article)

for i, article in enumerate(articles, 1):
    print(f"{i}. {article.published} - [{article.feed.name}] {article.title}")
    print(f"   {article.link}")
    print(f"   {article.tags}")
    print()


1. 2026-04-08 18:02:22+02:00 - [Email] Emerging from the Mythos
   https://feedyour.email/posts/2m5ef6qbl3ybjwycqhd1h79i
   []

2. 2026-04-08 17:48:39+02:00 - [Email] Your Daily Dose of European Tech News - Tech.eu (08/04/2026)
   https://feedyour.email/posts/e0jhpqz8svyuhwqmxs2g5atp
   []

3. 2026-04-08 17:04:56+02:00 - [Email] 😸 Strava for gambling
   https://feedyour.email/posts/b41hkorsstav56898vnwvnaf
   []

4. 2026-04-08 13:51:09+02:00 - [Email] Welcome to the free edition of Lenny's Newsletter 🥳
   https://feedyour.email/posts/tv4wac248r2fytyjkkzymcp0
   []

5. 2026-04-08 13:50:11+02:00 - [Email] You're on the list!
   https://feedyour.email/posts/q0l8qvs441ca4ejljqj2bbrc
   []

6. 2026-04-08 13:48:28+02:00 - [Email] 📫 Confirm your subscription to First Round
   https://feedyour.email/posts/zfywupn5tm003ijlm24esizo
   []

7. 2026-04-08 13:47:21+02:00 - [Email] Welcome to the free version of Refactoring!
   https://feedyour.email/posts/7hs9eag46bo5gnfuw63tcz8a
   []

8. 2026-04-0